# API 9: Videos

`plot()` (in `xkan.viz`) draws a KAN. To record the training dynamics, pass a `xkan.train.Figures` callback to `fit()` with the data to plot over, the image folder and the plot options. The callback saves the plot after each step as an image, and the images make the frames of a video.

In [1]:
import torch
from xkan import KANNetwork, create_dataset, fit
from xkan.train import Figures

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# create a KAN: 2D inputs, 1D output, and 5 hidden neurons. cubic spline (k=3), 3 grid intervals (grid=3).
model = KANNetwork([4,2,1,1], grid=3, k=3, seed=1, device=device)
f = lambda x: torch.exp((torch.sin(torch.pi*(x[:,[0]]**2+x[:,[1]]**2))+torch.sin(torch.pi*(x[:,[2]]**2+x[:,[3]]**2)))/2)
train, test = create_dataset(f, n_var=4, train_num=3000, device=device)

image_folder = 'video_img'

# train the model
#history = fit(model, train, test, steps=20, lamb=1e-3, lamb_entropy=2.)
figures = Figures(train['x'], folder=image_folder, beta=10,
                  in_vars=[r'$x_1$', r'$x_2$', r'$x_3$', r'$x_4$'],
                  out_vars=[r'${\rm exp}({\rm sin}(x_1^2+x_2^2)+{\rm sin}(x_3^2+x_4^2))$'])
history = fit(model, train, test, steps=5, lamb=0.001, lamb_entropy=2., callbacks=[figures])
print('train RMSE', history['train/y'][-1] ** 0.5, 'test RMSE', history['val/y'][-1] ** 0.5)

cuda


train RMSE 0.43026637185382366 test RMSE 0.4350245983743422


In [2]:
import os
import numpy as np
import moviepy.video.io.ImageSequenceClip # moviepy == 1.0.3

video_name='video'
fps=5

fps = fps
files = os.listdir(image_folder)
train_index = []
for file in files:
    if file[0].isdigit() and file.endswith('.jpg'):
        train_index.append(int(file[:-4]))

train_index = np.sort(train_index)

image_files = [image_folder+'/'+str(train_index[index])+'.jpg' for index in train_index]

clip = moviepy.video.io.ImageSequenceClip.ImageSequenceClip(image_files, fps=fps)
clip.write_videofile(video_name+'.mp4')

Moviepy - Building video video.mp4.
Moviepy - Writing video video.mp4

Moviepy - Done !
Moviepy - video ready video.mp4
